# Image Forensics — Detecting Tampered Images

## BEFORE clicking Run All — upload these files using the folder icon on the left sidebar

| File | Where to get it | Purpose |
|------|----------------|---------|
| `Cat_Test2.jpg` | Your `Data/` folder | Single-image demo |
| `dvmm_splc_uncompressed.zip` | Columbia dataset you downloaded | Batch evaluation |

**How to upload:**
1. Click the **folder icon** on the left sidebar in Colab
2. Click the **upload icon** (arrow pointing up)
3. Select both files
4. Wait for uploads to finish
5. Then click **Runtime → Run All**

Everything else (Python code, packages) is handled automatically by the notebook.

## Step 1 — Install Dependencies

In [ ]:
!pip install opencv-python-headless numpy scipy scikit-image scikit-learn matplotlib Pillow --quiet
print("All packages ready.")

## Step 2 — Write Detector Code
This cell creates `image_forensics.py` from code embedded in the notebook. No upload needed.

In [ ]:
%%writefile image_forensics.py
"""
Image Forensics Detector
========================
Detects image tampering on a SINGLE image (no reference needed — blind forensics).
Faithfully converts moonsandsk/Image-Forensics from MATLAB to Python, then
extends it with additional classical DIP techniques to cover the full course syllabus.

VERDICT: LIKELY TAMPERED / SUSPICIOUS / LIKELY AUTHENTIC
Based on weighted evidence fusion across all active detectors.

DIP Course Topics Covered (all 8 units):
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  1. Digital Image Fundamentals    → ELA, histogram analysis, CLAHE      │
  │  2. Image Transform & Enhancement → DFT, intensity transform, spatial   │
  │                                     filtering (Gaussian/Median/Laplacian)│
  │  3. Image & Video Coding          → JPEG artifact detection (8×8 DCT),  │
  │                                     copy-move detection (DCT patches)    │
  │  4. Image Restoration             → Multi-scale Wiener filter + K-means  │
  │  5. Color Image Processing        → RGB / HSV / YCbCr local analysis     │
  │  6. Morphological Processing      → Erosion, dilation, open, close,      │
  │                                     boundary, hole-fill, connected comps, │
  │                                     top-hat, grayscale morphology         │
  │  7. Image Segmentation            → Otsu threshold, Hough transform,     │
  │                                     watershed, edge detection             │
  │  8. Representation & Recognition  → GLCM Haralick features, boundary     │
  │                                     descriptors (area, perimeter,         │
  │                                     compactness, eccentricity)            │
  └─────────────────────────────────────────────────────────────────────────┘
"""

import os
import io
import warnings
import numpy as np
import cv2
from PIL import Image
from scipy.signal import wiener
from sklearn.cluster import KMeans
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import threshold_otsu
from skimage.segmentation import watershed
from skimage.measure import label, regionprops
from scipy import ndimage
import matplotlib
matplotlib.use("Agg")          # non-interactive backend; works on Colab too
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _norm(arr):
    arr = np.asarray(arr, dtype=np.float32)
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-8)


def _load(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot load: {path}")
    if img.shape[2] == 4:
        img = img[:, :, :3]
    return img


# ─────────────────────────────────────────────────────────────────────────────
# DETECTOR
# ─────────────────────────────────────────────────────────────────────────────

class ImageForensicsDetector:
    """
    Single-image blind tampering detector.
    No reference image required — detects internal inconsistencies.
    """

    def __init__(self, image_path: str):
        bgr = _load(image_path)
        self.path      = image_path
        self.bgr       = bgr
        self.rgb       = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        self.gray_u8   = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        self.gray      = self.gray_u8.astype(np.float64) / 255.0
        self.results   = {}
        self.scores    = {}     # 0-1 suspicion score per technique
        self.detected  = {}     # bool per technique

    # ═════════════════════════════════════════════════════════════════════════
    # 1. ELA — Error Level Analysis
    #    DIP Topics: Digital Image Fundamentals, Image & Video Coding
    # ═════════════════════════════════════════════════════════════════════════
    def detect_ela(self, quality: int = 95):
        """
        Save the image as JPEG at a given quality, reload it, and compute
        the absolute difference. Authentic regions degrade uniformly; spliced
        regions that were already compressed once show anomalously LOW error,
        while freshly added regions may show anomalously HIGH error.
        """
        buf = io.BytesIO()
        pil_img = Image.fromarray(self.rgb)
        pil_img.save(buf, format="JPEG", quality=quality)
        buf.seek(0)
        recomp = np.array(Image.open(buf).convert("RGB")).astype(np.float32) / 255.0
        orig   = self.rgb.astype(np.float32) / 255.0

        ela_map  = np.abs(orig - recomp)
        ela_gray = ela_map.mean(axis=2)

        # Contrast-enhance to make subtle differences visible
        ela_eq   = cv2.equalizeHist((ela_gray * 255).astype(np.uint8))
        ela_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(ela_eq)
        ela_vis  = ela_clahe.astype(np.float32) / 255.0

        mu, sigma = ela_gray.mean(), ela_gray.std()
        thresh    = mu + 2 * sigma
        n_suspicious = int(np.sum(ela_gray > thresh))
        suspicion_pct = n_suspicious / ela_gray.size * 100
        score = min(suspicion_pct / 20.0, 1.0)   # normalise: 20% → score=1

        self.results["ela"] = dict(map=ela_map, gray=ela_gray, vis=ela_vis,
                                   mean=mu, std=sigma, suspicion_pct=suspicion_pct)
        self.scores["ela"]   = score
        self.detected["ela"] = score > 0.25
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 2. JPEG Artifact Detection — 8×8 DCT block analysis
    #    DIP Topics: Image & Video Coding, Image Transformation
    # ═════════════════════════════════════════════════════════════════════════
    def detect_jpeg_artifacts(self):
        """
        JPEG splits images into 8×8 blocks in YCbCr space. Tampering (paste,
        resize, re-save) disrupts the standard block statistics.
        Analyses luminance-channel block standard deviation and internal edge
        density; anomalous blocks are flagged.
        """
        ycbcr  = cv2.cvtColor(self.bgr, cv2.COLOR_BGR2YCrCb)
        Y      = ycbcr[:, :, 0].astype(np.float64)
        h, w   = Y.shape
        bs     = 8
        rows   = h // bs
        cols   = w // bs

        std_map  = np.zeros((rows, cols), dtype=np.float32)
        edge_map = np.zeros((rows, cols), dtype=np.float32)

        for r in range(rows):
            for c in range(cols):
                blk = Y[r*bs:(r+1)*bs, c*bs:(c+1)*bs]
                std_map[r, c]  = float(np.std(blk))
                edge_map[r, c] = float(np.mean(np.abs(np.diff(blk))))

        std_up   = cv2.resize(std_map,  (w, h), interpolation=cv2.INTER_NEAREST)
        edge_up  = cv2.resize(edge_map, (w, h), interpolation=cv2.INTER_NEAREST)
        artifact = _norm(std_up) + _norm(edge_up)
        artifact = _norm(artifact)
        artifact = cv2.medianBlur((artifact * 255).astype(np.uint8), 3).astype(np.float32) / 255.0

        # Otsu threshold on artifact map (Image Segmentation — Otsu's method)
        otsu_val, suspicious = cv2.threshold(
            (artifact * 255).astype(np.uint8), 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        suspicious = suspicious.astype(bool)
        score = float(np.sum(suspicious)) / suspicious.size

        self.results["jpeg"] = dict(std_map=std_up, edge_map=edge_up,
                                    artifact=artifact, suspicious=suspicious,
                                    otsu_val=otsu_val / 255.0)
        self.scores["jpeg"]   = min(score / 0.3, 1.0)
        self.detected["jpeg"] = score > 0.15
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 3. Noise Inconsistency — Multi-scale Wiener + K-means
    #    DIP Topics: Image Restoration (Wiener filter), Image Fundamentals
    # ═════════════════════════════════════════════════════════════════════════
    def detect_noise_inconsistency(self):
        """
        Applies Wiener denoising at three different kernel sizes (3,5,7).
        The noise residue = original − denoised reveals the camera-noise pattern.
        Different cameras impart different noise signatures; spliced objects
        carry "foreign" noise. K-means (k=3) clusters the noise map to separate
        the background noise level from anomalous regions.
        """
        g = self.gray
        noise_maps = []
        for ks in [3, 5, 7]:
            denoised = wiener(g, mysize=ks)
            noise_maps.append(np.abs(g - denoised))

        noise_map = _norm(np.mean(noise_maps, axis=0))

        pixels = noise_map.flatten().reshape(-1, 1)
        pixels = pixels + np.random.randn(*pixels.shape) * 0.001  # avoid flat crash
        km = KMeans(n_clusters=3, n_init=5, max_iter=200, random_state=42)
        km.fit(pixels)
        labels = km.labels_.reshape(noise_map.shape)

        # Suspicious cluster = the one with the highest noise mean
        cluster_means = [float(noise_map[labels == k].mean()) for k in range(3)]
        suspect_k = int(np.argmax(cluster_means))
        suspect_mask = (labels == suspect_k)

        score = float(np.sum(suspect_mask)) / suspect_mask.size

        self.results["noise"] = dict(noise_map=noise_map, clusters=labels,
                                     suspect_mask=suspect_mask)
        self.scores["noise"]   = min(score / 0.3, 1.0)
        self.detected["noise"] = score > 0.12
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 4. Edge Inconsistency + Hough Transform
    #    DIP Topics: Image Segmentation (edge detection, Hough transform)
    #                Image Transformation & Enhancement
    # ═════════════════════════════════════════════════════════════════════════
    def detect_edge_and_hough(self):
        """
        Computes gradient magnitude using Sobel and detects edges with
        Canny, Sobel, and Prewitt operators. Edges whose magnitude deviates
        more than 2σ from the median are considered "suspicious" — these often
        correspond to pasted boundaries or double-compressed regions.

        Also applies Probabilistic Hough Line Transform (Image Segmentation):
        a tampered image may show unnaturally high numbers of detected lines
        (geometric artifacts from splicing or resizing).
        """
        g8 = self.gray_u8

        # ── Edge maps ────────────────────────────────────────────────────────
        canny   = cv2.Canny(g8, 50, 150)
        sobelx  = cv2.Sobel(g8, cv2.CV_64F, 1, 0, ksize=3)
        sobely  = cv2.Sobel(g8, cv2.CV_64F, 0, 1, ksize=3)
        mag     = np.sqrt(sobelx**2 + sobely**2)

        kx      = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], np.float32)
        ky      = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], np.float32)
        prewitt = np.sqrt(cv2.filter2D(g8.astype(np.float32),-1,kx)**2 +
                          cv2.filter2D(g8.astype(np.float32),-1,ky)**2)

        edge_combined = ((canny > 0) |
                         (mag > mag.mean() + mag.std()) |
                         (prewitt > prewitt.mean() + prewitt.std()))

        # Suspicious edges = magnitude far from image's own median
        edge_vals = mag[edge_combined]
        if len(edge_vals) > 0:
            med, sd  = float(np.median(edge_vals)), float(np.std(edge_vals))
            suspicious_edges = edge_combined & (np.abs(mag - med) > 2 * sd)
            suspicious_edges = cv2.morphologyEx(
                suspicious_edges.astype(np.uint8), cv2.MORPH_OPEN,
                np.ones((3, 3), np.uint8)).astype(bool)
        else:
            suspicious_edges = np.zeros_like(edge_combined)

        edge_score = float(np.sum(suspicious_edges)) / suspicious_edges.size

        # ── Hough Transform (Image Segmentation) ─────────────────────────────
        lines = cv2.HoughLinesP(canny, 1, np.pi / 180,
                                threshold=80, minLineLength=50, maxLineGap=10)
        hough_img = cv2.cvtColor(g8, cv2.COLOR_GRAY2RGB)
        n_lines   = 0
        if lines is not None:
            n_lines = len(lines)
            for ln in lines:
                x1, y1, x2, y2 = ln[0]
                cv2.line(hough_img, (x1, y1), (x2, y2), (255, 60, 60), 1)

        # Unusually many detected lines → possible geometric manipulation
        area_k = (self.gray.shape[0] * self.gray.shape[1]) / 1e5
        hough_score = min(n_lines / max(area_k * 30, 1), 1.0)

        score = (edge_score * 0.6 + hough_score * 0.4)

        self.results["edge"] = dict(canny=canny, mag=_norm(mag),
                                    prewitt=_norm(prewitt),
                                    suspicious=suspicious_edges,
                                    hough_img=hough_img, n_lines=n_lines)
        self.scores["edge"]   = min(score / 0.2, 1.0)
        self.detected["edge"] = score > 0.08
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 5. Copy-Move Detection — DCT Patch Matching
    #    DIP Topics: Image & Video Coding, Representation & Recognition
    # ═════════════════════════════════════════════════════════════════════════
    def detect_copy_move(self, patch_size: int = 16, step: int = 8):
        """
        Divides the image into overlapping patches, extracts their low-frequency
        DCT coefficients (compact, illumination-robust features), and searches for
        identical patches that are far apart — the signature of cloning/copy-move.
        """
        img = self.gray_u8
        h, w = img.shape
        patches, coords = [], []

        for y in range(0, h - patch_size + 1, step):
            for x in range(0, w - patch_size + 1, step):
                blk  = img[y:y+patch_size, x:x+patch_size].astype(np.float32)
                dct  = cv2.dct(blk)
                feat = dct[:8, :8].flatten()
                patches.append(feat)
                coords.append((y, x))

        if len(patches) < 2:
            self.results["copy_move"] = dict(map=np.zeros_like(img, bool))
            self.scores["copy_move"] = self.detected["copy_move"] = 0
            return 0

        data = np.array(patches, dtype=np.float32)
        mu, sd = data.mean(1, keepdims=True), data.std(1, keepdims=True)
        sd[sd < 1e-10] = 1.0
        data = (data - mu) / sd

        # Correlation matrix (memory-efficient: O(n·f) not O(n²) in RAM)
        C = data @ data.T / data.shape[1]
        corr_vals = C[np.triu_indices_from(C, k=1)]
        thresh    = max(float(corr_vals.mean() + 3 * corr_vals.std()), 0.95)

        cm_map      = np.zeros((h, w), dtype=np.uint8)
        matched     = 0
        coords_arr  = np.array(coords)

        for i in range(len(patches)):
            js = np.where((C[i] > thresh))[0]
            js = js[js > i]
            for j in js:
                y1, x1 = coords[i]
                y2, x2 = coords[j]
                if np.sqrt((y1-y2)**2 + (x1-x2)**2) > patch_size * 2:
                    cm_map[y1:y1+patch_size, x1:x1+patch_size] = 255
                    cm_map[y2:y2+patch_size, x2:x2+patch_size] = 255
                    matched += 1

        # Clean up (Morphological closing to fill gaps + remove noise)
        cm_map = cv2.morphologyEx(cm_map, cv2.MORPH_CLOSE,
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
        # Remove tiny regions (speckle noise) — uses connected component analysis
        n_labels, label_map, stats, _ = cv2.connectedComponentsWithStats(cm_map)
        for lbl in range(1, n_labels):
            if stats[lbl, cv2.CC_STAT_AREA] < 200:
                cm_map[label_map == lbl] = 0

        score = min(matched / 20.0, 1.0)

        self.results["copy_move"] = dict(map=cm_map.astype(bool), matched=matched)
        self.scores["copy_move"]   = score
        self.detected["copy_move"] = matched > 3
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 6. DFT Frequency Domain Analysis
    #    DIP Topics: Image Transformation & Enhancement (DFT, frequency filtering)
    # ═════════════════════════════════════════════════════════════════════════
    def detect_dft_anomalies(self):
        """
        Computes the 2D DFT of the grayscale image.
        Periodic noise (from resizing, double-JPEG, or copy-paste) appears as
        sharp peaks in the frequency spectrum. Detects these by comparing
        the spectrum against a smooth radial baseline.
        """
        fft    = np.fft.fft2(self.gray)
        fft_sh = np.fft.fftshift(fft)
        mag    = np.log1p(np.abs(fft_sh))
        phase  = np.angle(fft_sh)

        # Build radial average baseline (what a "normal" spectrum looks like)
        h, w   = mag.shape
        cy, cx = h // 2, w // 2
        Y, X   = np.ogrid[:h, :w]
        R      = np.sqrt((X - cx)**2 + (Y - cy)**2).astype(int)
        R      = np.clip(R, 0, min(cy, cx) - 1)
        radial_mean = np.array([mag[R == r].mean() for r in range(R.max() + 1)])
        baseline    = radial_mean[R]

        # Residual = deviation from expected radially-symmetric spectrum
        residual    = _norm(np.abs(mag - baseline))

        # Sharp peaks in the residual → periodic noise → tampering
        peak_thresh = float(residual.mean() + 2 * residual.std())
        peaks       = residual > peak_thresh
        score       = float(np.sum(peaks)) / peaks.size

        self.results["dft"] = dict(magnitude=_norm(mag), phase=_norm(np.abs(phase)),
                                   residual=residual, peaks=peaks)
        self.scores["dft"]   = min(score / 0.05, 1.0)
        self.detected["dft"] = score > 0.02
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 7. Histogram Analysis + Intensity Transform
    #    DIP Topics: Digital Image Fundamentals, Image Transform & Enhancement
    # ═════════════════════════════════════════════════════════════════════════
    def detect_histogram_anomalies(self):
        """
        Analyses the global histogram for manipulation artifacts:
          - Comb effect: periodic zero-bins after brightness/contrast adjustments
          - Clipping: excessive pixels at 0 or 255 (over-exposure / over-editing)
          - Histogram spikes: unusual sharp peaks

        Also applies:
          - Log transform  (reveals detail in dark regions)
          - Gamma correction (γ=0.5 — boosts mid-tone tamper artifacts)
          - CLAHE (adaptive histogram equalization for local enhancement)
        """
        g = self.gray_u8
        hist = cv2.calcHist([g], [0], None, [256], [0, 256]).flatten()

        # Comb detection: count empty bins between non-empty bins
        nonzero    = (hist > 0).astype(int)
        runs       = np.diff(nonzero)
        n_gaps     = int(np.sum(runs == -1))
        gap_ratio  = n_gaps / 128.0

        # Clipping
        clip_lo  = float(hist[:5].sum())  / hist.sum()
        clip_hi  = float(hist[-5:].sum()) / hist.sum()
        clip_score = max(clip_lo, clip_hi)

        # Log transform (Image Transformation)
        log_img  = _norm(np.log1p(self.gray.astype(np.float64)))

        # Gamma correction (Image Transformation)
        gamma    = 0.5
        gamma_img = np.power(np.clip(self.gray, 0, 1), gamma)

        # CLAHE (Image Enhancement — adaptive histogram equalization)
        clahe     = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(g)

        score = float(gap_ratio * 0.5 + clip_score * 0.5)

        self.results["histogram"] = dict(hist=hist, log_img=log_img,
                                         gamma_img=gamma_img, clahe_img=clahe_img,
                                         gap_ratio=gap_ratio, clip_score=clip_score)
        self.scores["histogram"]   = min(score, 1.0)
        self.detected["histogram"] = score > 0.15
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 8. Spatial Filtering
    #    DIP Topics: Image Transformation & Enhancement (spatial domain filtering)
    # ═════════════════════════════════════════════════════════════════════════
    def detect_spatial_anomalies(self):
        """
        Applies Gaussian, Median, and Laplacian (sharpening) filters.
        Analyses the noise residual (original − smooth) and the Laplacian
        response for local inconsistencies — tampered regions often show
        anomalous sharpness or blurriness relative to the rest of the image.
        """
        g8 = self.gray_u8
        gf = self.gray.astype(np.float32)

        gauss    = cv2.GaussianBlur(gf, (5, 5), 0)
        median   = cv2.medianBlur(g8, 5).astype(np.float32) / 255.0
        laplacian = cv2.Laplacian(g8, cv2.CV_64F)

        noise_gauss  = np.abs(gf - gauss)
        noise_median = np.abs(gf - median)

        # Local variance of Laplacian → sharpness map
        blur_local = cv2.GaussianBlur(laplacian.astype(np.float32), (15, 15), 0)
        lap_var    = np.abs(laplacian - blur_local.astype(np.float64))

        # Regions with very different sharpness from their neighbourhood
        lap_n    = _norm(lap_var)
        mean_lap = float(lap_n.mean())
        std_lap  = float(lap_n.std())
        anomalous_sharpness = lap_n > (mean_lap + 2.5 * std_lap)

        score = float(np.sum(anomalous_sharpness)) / anomalous_sharpness.size

        self.results["spatial"] = dict(gauss_residual=_norm(noise_gauss),
                                       median_residual=_norm(noise_median),
                                       laplacian=_norm(np.abs(laplacian)),
                                       sharpness_anom=anomalous_sharpness)
        self.scores["spatial"]   = min(score / 0.05, 1.0)
        self.detected["spatial"] = score > 0.03
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 9. Color Space Analysis
    #    DIP Topics: Color Image Processing (RGB, HSV, YCbCr)
    # ═════════════════════════════════════════════════════════════════════════
    def detect_color_anomalies(self):
        """
        Analyses local colour statistics in sliding 32×32 windows across
        three colour spaces (RGB, HSV, YCbCr). A tampered region pasted from a
        different camera or under different lighting will show inconsistent
        hue, saturation, or chrominance statistics relative to its surroundings.
        """
        rgb  = self.rgb.astype(np.float32) / 255.0
        hsv  = cv2.cvtColor(self.bgr, cv2.COLOR_BGR2HSV).astype(np.float32) / np.array([180,255,255])
        ycbcr = cv2.cvtColor(self.bgr, cv2.COLOR_BGR2YCrCb).astype(np.float32) / 255.0

        h, w = self.gray.shape
        win  = 32
        step = 16
        rows = (h - win) // step
        cols = (w - win) // step

        # Store per-window mean for each channel
        rgb_means, hsv_means, ycc_means = [], [], []

        for i in range(rows):
            for j in range(cols):
                r0, r1 = i * step, i * step + win
                c0, c1 = j * step, j * step + win
                rgb_means.append(rgb[r0:r1, c0:c1].mean(axis=(0, 1)))
                hsv_means.append(hsv[r0:r1, c0:c1].mean(axis=(0, 1)))
                ycc_means.append(ycbcr[r0:r1, c0:c1].mean(axis=(0, 1)))

        rgb_means = np.array(rgb_means)
        hsv_means = np.array(hsv_means)
        ycc_means = np.array(ycc_means)

        # Z-score each channel; outlier windows = potential splice boundaries
        def z_outlier_ratio(arr):
            z = (arr - arr.mean(0)) / (arr.std(0) + 1e-8)
            return float((np.abs(z) > 2.5).any(axis=1).mean())

        rgb_score = z_outlier_ratio(rgb_means)
        hsv_score = z_outlier_ratio(hsv_means)
        ycc_score = z_outlier_ratio(ycc_means)
        score     = (rgb_score + hsv_score + ycc_score) / 3

        # Build visualisation maps
        hue_ch = hsv[:, :, 0]
        sat_ch = hsv[:, :, 1]
        cb_ch  = ycbcr[:, :, 2]
        cr_ch  = ycbcr[:, :, 1]

        self.results["color"] = dict(hue=hue_ch, sat=sat_ch, cb=cb_ch, cr=cr_ch,
                                     rgb_score=rgb_score, hsv_score=hsv_score,
                                     ycc_score=ycc_score)
        self.scores["color"]   = min(score / 0.3, 1.0)
        self.detected["color"] = score > 0.12
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 10. Morphological Analysis
    #     DIP Topics: Morphological Image Processing
    #     (erosion, dilation, open, close, boundary, hole-fill, connected
    #      components, top-hat, grayscale morphology, thinning)
    # ═════════════════════════════════════════════════════════════════════════
    def detect_morphological_anomalies(self):
        """
        Applies the full suite of morphological operations to the binarised image
        and analyses the properties of connected components. Tampered regions
        often produce abnormally large or abnormally compact connected components,
        or leave unusual holes that the authentic background would not have.
        """
        g8     = self.gray_u8
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

        # Otsu binarisation (Image Segmentation)
        otsu_val = threshold_otsu(g8)
        binary   = (g8 > otsu_val).astype(np.uint8) * 255

        # Core morphological operations
        eroded   = cv2.erode(binary, kernel)
        dilated  = cv2.dilate(binary, kernel)
        opened   = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel)
        closed   = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

        # Boundary extraction = binary − eroded
        boundary = cv2.subtract(binary, eroded)

        # Hole filling using flood-fill from corners
        filled   = binary.copy()
        mask_ff  = np.zeros((g8.shape[0] + 2, g8.shape[1] + 2), np.uint8)
        cv2.floodFill(filled, mask_ff, (0, 0), 255)
        filled   = cv2.bitwise_not(filled)
        filled   = cv2.bitwise_or(binary, filled)

        # Grayscale morphological operations (top-hat / black-hat)
        top_hat   = cv2.morphologyEx(g8, cv2.MORPH_TOPHAT,   kernel)   # bright details
        black_hat = cv2.morphologyEx(g8, cv2.MORPH_BLACKHAT, kernel)   # dark details
        morph_grad = cv2.morphologyEx(g8, cv2.MORPH_GRADIENT, kernel)  # edges

        # Skeletonisation (thinning) via Zhang-Suen approximation
        skel  = cv2.ximgproc.thinning(binary) if hasattr(cv2, 'ximgproc') else boundary

        # Connected component analysis
        n_labels, label_map, stats, centroids = cv2.connectedComponentsWithStats(binary)
        areas      = stats[1:, cv2.CC_STAT_AREA]          # skip background
        perims_raw = []
        compact    = []
        for lbl in range(1, n_labels):
            mask = (label_map == lbl).astype(np.uint8)
            cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            p = cv2.arcLength(cnts[0], True) if cnts else 0
            perims_raw.append(p)
            a = stats[lbl, cv2.CC_STAT_AREA]
            compact.append((4 * np.pi * a / (p**2 + 1e-8)) if p > 0 else 1.0)

        areas   = np.array(areas, dtype=np.float32)
        compact = np.array(compact, dtype=np.float32)

        # Outlier components (unusually large area or unusual compactness)
        area_z    = np.abs((areas - areas.mean()) / (areas.std() + 1e-8))
        cmp_z     = np.abs((compact - compact.mean()) / (compact.std() + 1e-8))
        n_outliers = int(np.sum((area_z > 2.5) | (cmp_z > 2.5)))
        score      = min(n_outliers / max(len(areas) * 0.1, 1), 1.0)

        self.results["morph"] = dict(
            binary=binary, eroded=eroded, dilated=dilated,
            opened=opened, closed=closed, boundary=boundary,
            filled=filled, top_hat=top_hat, black_hat=black_hat,
            morph_grad=morph_grad, label_map=label_map,
            n_labels=n_labels, areas=areas, compact=compact
        )
        self.scores["morph"]   = float(score)
        self.detected["morph"] = n_outliers > 2
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 11. Watershed Segmentation + Region Analysis
    #     DIP Topics: Image Segmentation (watershed, region-based)
    # ═════════════════════════════════════════════════════════════════════════
    def detect_watershed_regions(self):
        """
        Uses the watershed algorithm to segment the image into regions.
        Analyses region properties (area, eccentricity, mean intensity) for
        statistical outliers. A pasted object typically creates a region with
        a very different texture or intensity from its surroundings.
        """
        g8   = self.gray_u8

        # Compute distance transform from foreground pixels
        _, binary_fg = cv2.threshold(g8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        dist    = ndimage.distance_transform_edt(binary_fg)

        # Local maxima → seeds (markers)
        markers = np.zeros_like(g8, dtype=np.int32)
        kernel  = np.ones((11, 11), np.uint8)
        local_max = cv2.dilate(dist.astype(np.float32), kernel) == dist
        markers[local_max & (dist > dist.max() * 0.2)] = 1
        markers, _ = ndimage.label(markers)

        # Run watershed on the gradient image
        gradient = cv2.morphologyEx(g8, cv2.MORPH_GRADIENT,
                                    np.ones((3, 3), np.uint8))
        seg = watershed(gradient, markers, mask=binary_fg.astype(bool))

        # Analyse region properties
        props   = regionprops(seg, intensity_image=g8)
        if len(props) == 0:
            score = 0.0
        else:
            intensities  = np.array([p.mean_intensity for p in props])
            eccentricitiies = np.array([p.eccentricity for p in props])
            areas_w      = np.array([p.area for p in props])

            int_z  = np.abs((intensities - intensities.mean()) / (intensities.std() + 1e-8))
            ecc_z  = np.abs((eccentricitiies - eccentricitiies.mean()) / (eccentricitiies.std() + 1e-8))
            area_z = np.abs((areas_w - areas_w.mean()) / (areas_w.std() + 1e-8))
            outliers = int((int_z > 2.5).sum() + (ecc_z > 2.5).sum() + (area_z > 2.5).sum())
            score    = min(outliers / max(len(props), 1), 1.0)

        # Colour the segmentation map for visualisation
        n_seg   = int(seg.max())
        seg_vis = np.zeros((*seg.shape, 3), dtype=np.uint8)
        for lbl in range(1, n_seg + 1):
            c = np.random.RandomState(lbl).randint(50, 255, 3)
            seg_vis[seg == lbl] = c

        self.results["watershed"] = dict(seg=seg, seg_vis=seg_vis,
                                         gradient=gradient, dist=_norm(dist))
        self.scores["watershed"]   = float(score)
        self.detected["watershed"] = score > 0.2
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # 12. GLCM Texture + Boundary Descriptors
    #     DIP Topics: Representation, Description & Recognition of Objects
    # ═════════════════════════════════════════════════════════════════════════
    def detect_texture_and_boundaries(self):
        """
        Computes GLCM Haralick features (Contrast, Correlation, Energy,
        Homogeneity) in sliding windows. Windows with feature vectors far from
        the image's global distribution are flagged as anomalous.

        Also computes boundary descriptors for the largest connected component:
        area, perimeter, compactness, eccentricity — used in Representation &
        Recognition to characterise regions.
        """
        g8   = (self.gray * 255 / 4).astype(np.uint8)   # quantise to 64 levels
        h, w = g8.shape
        win, step = 48, 24

        feats = []
        for i in range(0, h - win, step):
            for j in range(0, w - win, step):
                patch = g8[i:i+win, j:j+win]
                glcm  = graycomatrix(patch, [1],
                                     [0, np.pi/4, np.pi/2, 3*np.pi/4],
                                     levels=64, symmetric=True, normed=True)
                feats.append([
                    float(graycoprops(glcm, "contrast").mean()),
                    float(graycoprops(glcm, "correlation").mean()),
                    float(graycoprops(glcm, "energy").mean()),
                    float(graycoprops(glcm, "homogeneity").mean()),
                ])

        feats = np.array(feats)
        z     = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)
        outlier_windows = float((np.abs(z) > 2.5).any(axis=1).mean())

        # ── Boundary Descriptors ─────────────────────────────────────────────
        _, binary = cv2.threshold(self.gray_u8, 0, 255, cv2.THRESH_OTSU)
        cnts, _   = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        descriptors = []
        for cnt in cnts:
            area   = float(cv2.contourArea(cnt))
            perim  = float(cv2.arcLength(cnt, True))
            if area < 200 or perim < 1:
                continue
            compact = 4 * np.pi * area / (perim ** 2)
            bb_area = float(cv2.boundingRect(cnt)[2] * cv2.boundingRect(cnt)[3])
            extent  = area / max(bb_area, 1)
            descriptors.append(dict(area=area, perimeter=perim,
                                    compactness=compact, extent=extent))

        # Contour visualisation
        contour_vis = self.rgb.copy()
        cv2.drawContours(contour_vis, cnts, -1, (255, 60, 60), 1)

        score = min(outlier_windows / 0.3, 1.0)

        self.results["glcm"] = dict(feats=feats,
                                    names=["Contrast","Correlation","Energy","Homogeneity"],
                                    descriptors=descriptors,
                                    contour_vis=contour_vis,
                                    outlier_ratio=outlier_windows)
        self.scores["glcm"]   = score
        self.detected["glcm"] = outlier_windows > 0.15
        return score

    # ═════════════════════════════════════════════════════════════════════════
    # RUN ALL
    # ═════════════════════════════════════════════════════════════════════════
    def run_all(self, verbose: bool = True) -> tuple:
        """
        Runs all 12 forensic analyses. Returns (verdict, confidence, scores).
        """
        steps = [
            ("ela",       "ELA — Error Level Analysis",                       self.detect_ela),
            ("jpeg",      "JPEG Artifact Detection (8×8 DCT, YCbCr, Otsu)",   self.detect_jpeg_artifacts),
            ("noise",     "Noise Inconsistency (Wiener ×3 + K-means)",         self.detect_noise_inconsistency),
            ("edge",      "Edge Detection (Canny/Sobel/Prewitt) + Hough",     self.detect_edge_and_hough),
            ("copy_move", "Copy-Move Detection (DCT patch matching)",          self.detect_copy_move),
            ("dft",       "DFT Frequency Spectrum Analysis",                   self.detect_dft_anomalies),
            ("histogram", "Histogram + Intensity Transform (log/gamma/CLAHE)", self.detect_histogram_anomalies),
            ("spatial",   "Spatial Filtering (Gaussian/Median/Laplacian)",     self.detect_spatial_anomalies),
            ("color",     "Color Space Analysis (RGB / HSV / YCbCr)",          self.detect_color_anomalies),
            ("morph",     "Morphological Analysis (full pipeline)",            self.detect_morphological_anomalies),
            ("watershed", "Watershed + Region Segmentation",                   self.detect_watershed_regions),
            ("glcm",      "GLCM Texture + Boundary Descriptors",               self.detect_texture_and_boundaries),
        ]

        for i, (key, label, fn) in enumerate(steps, 1):
            if verbose:
                print(f"  [{i:02d}/{len(steps)}] {label}...")
            try:
                fn()
            except Exception as e:
                if verbose:
                    print(f"         ⚠ skipped ({e})")
                self.scores[key]   = 0.0
                self.detected[key] = False

        # Evidence fusion — weighted average of suspicion scores
        weights = {"ela":       0.20, "jpeg":    0.15, "noise":    0.15,
                   "edge":      0.10, "copy_move":0.10, "dft":     0.05,
                   "histogram": 0.05, "spatial":  0.05, "color":   0.05,
                   "morph":     0.03, "watershed":0.04, "glcm":    0.03}

        weighted_score = sum(self.scores.get(k, 0) * w for k, w in weights.items())
        n_triggered    = sum(self.detected.values())
        total          = len(self.detected)

        if weighted_score > 0.55 or n_triggered >= 6:
            verdict = "LIKELY TAMPERED"
        elif weighted_score > 0.30 or n_triggered >= 4:
            verdict = "SUSPICIOUS"
        else:
            verdict = "LIKELY AUTHENTIC"

        confidence = weighted_score * 100

        if verbose:
            colour = {"LIKELY TAMPERED": "⚠ ", "SUSPICIOUS": "? ", "LIKELY AUTHENTIC": "✓ "}
            label_w = {"ela":"ELA", "jpeg":"JPEG Artifacts", "noise":"Wiener Noise",
                       "edge":"Edge+Hough", "copy_move":"Copy-Move", "dft":"DFT Spectrum",
                       "histogram":"Histogram", "spatial":"Spatial Filter",
                       "color":"Color Spaces", "morph":"Morphology",
                       "watershed":"Watershed", "glcm":"GLCM Texture"}
            print(f"\n{'═'*58}")
            print(f"  VERDICT   : {colour[verdict]}{verdict}")
            print(f"  Suspicion : {confidence:.1f} / 100")
            print(f"  Triggered : {n_triggered}/{total} detectors")
            print(f"{'═'*58}")
            for k, w in weights.items():
                flag = "✗" if self.detected.get(k) else "✓"
                sc   = self.scores.get(k, 0)
                print(f"  {flag} {label_w[k]:<20s}  score={sc:.3f}  weight={w:.2f}")
            print(f"{'═'*58}\n")

        return verdict, confidence, self.scores

    # ═════════════════════════════════════════════════════════════════════════
    # DASHBOARD
    # ═════════════════════════════════════════════════════════════════════════
    def plot_dashboard(self, save_path: str = None, show: bool = True):
        """Render full forensic dashboard (12 techniques, ~30 sub-plots)."""
        if not self.scores:
            raise RuntimeError("Call run_all() first.")

        verdict    = ("LIKELY TAMPERED"  if self.scores.get("ela",0)*0.2+self.scores.get("jpeg",0)*0.15 > 0.15
                      else "SUSPICIOUS"  if sum(self.detected.values()) >= 4
                      else "LIKELY AUTHENTIC")
        # recalculate properly
        weights    = {"ela":0.20,"jpeg":0.15,"noise":0.15,"edge":0.10,"copy_move":0.10,
                      "dft":0.05,"histogram":0.05,"spatial":0.05,"color":0.05,
                      "morph":0.03,"watershed":0.04,"glcm":0.03}
        ws         = sum(self.scores.get(k,0)*w for k,w in weights.items())
        n_trig     = sum(self.detected.values())
        verdict    = ("LIKELY TAMPERED"  if ws > 0.55 or n_trig >= 6 else
                      "SUSPICIOUS"       if ws > 0.30 or n_trig >= 4 else
                      "LIKELY AUTHENTIC")
        vcolour    = {"LIKELY TAMPERED":"#ff4444","SUSPICIOUS":"#ffaa00","LIKELY AUTHENTIC":"#44ff88"}[verdict]

        fig = plt.figure(figsize=(28, 40))
        fig.patch.set_facecolor("#0d0d0d")

        fig.text(0.5, 0.992, "IMAGE FORENSICS DASHBOARD",
                 ha="center", va="top", fontsize=22, color="white",
                 fontweight="bold", fontfamily="monospace")
        fig.text(0.5, 0.984,
                 f"Verdict: {verdict}   |   suspicion={ws*100:.1f}/100   |   {n_trig}/12 detectors triggered",
                 ha="center", va="top", fontsize=13, color=vcolour, fontweight="bold")

        gs = gridspec.GridSpec(14, 4, figure=fig, hspace=0.6, wspace=0.25,
                               top=0.978, bottom=0.010, left=0.04, right=0.98)

        def _ax(r, c, cs=1): return fig.add_subplot(gs[r, c:c+cs])

        def _show(ax, img, title, cmap=None):
            ax.set_facecolor("#1c1c1c")
            ax.imshow(_norm(img) if img.ndim == 2 else img, cmap=cmap or "gray",
                      aspect="auto", interpolation="nearest")
            ax.set_title(title, fontsize=7, color="white", pad=2)
            ax.axis("off")

        def _flag(ax, key):
            det = self.detected.get(key, False)
            ax.text(0.5, -0.08, "⚠ SUSPICIOUS" if det else "✓ CLEAR",
                    transform=ax.transAxes, ha="center", fontsize=6.5,
                    color="#ff6666" if det else "#66ff99", fontweight="bold")

        # Row 0: original
        _show(_ax(0,0,2), self.rgb,     "Input Image (RGB)")
        _show(_ax(0,2),   self.gray_u8, "Grayscale")
        r = self.results.get("ela", {})
        if r:
            _show(_ax(0,3), r["vis"], "ELA (enhanced)", "hot")
            _flag(_ax(0,3), "ela")

        # Row 1: ELA detail + JPEG
        if r:
            _show(_ax(1,0), r["gray"],   "ELA Raw Map", "hot")
        r2 = self.results.get("jpeg", {})
        if r2:
            _show(_ax(1,1), r2["artifact"],   "JPEG Artifact Map")
            _show(_ax(1,2), r2["std_map"],    "Block Std (8×8)")
            _show(_ax(1,3), r2["suspicious"], "Suspicious Blocks")
            _flag(_ax(1,3), "jpeg")

        # Row 2: Noise
        r = self.results.get("noise", {})
        if r:
            _show(_ax(2,0), r["noise_map"],   "Noise Map (Wiener)")
            _show(_ax(2,1,3), r["clusters"],  "K-means Clusters", "tab10")
            _flag(_ax(2,1,3), "noise")

        # Row 3: Edge + Hough
        r = self.results.get("edge", {})
        if r:
            _show(_ax(3,0), r["canny"],      "Canny Edges")
            _show(_ax(3,1), r["mag"],        "Sobel Magnitude")
            _show(_ax(3,2), r["prewitt"],    "Prewitt")
            _show(_ax(3,3), r["hough_img"],  f"Hough Lines (n={r['n_lines']})")
            _flag(_ax(3,3), "edge")

        # Row 4: Copy-move + suspicious edges
        r = self.results.get("edge", {})
        if r:
            _show(_ax(4,0), r["suspicious"], "Suspicious Edges", "hot")
        r = self.results.get("copy_move", {})
        if r:
            _show(_ax(4,1,3), r["map"].astype(np.uint8)*255, "Copy-Move Map", "hot")
            _flag(_ax(4,1,3), "copy_move")

        # Row 5: DFT
        r = self.results.get("dft", {})
        if r:
            _show(_ax(5,0), r["magnitude"], "DFT Magnitude", "inferno")
            _show(_ax(5,1), r["phase"],     "DFT Phase",     "twilight")
            _show(_ax(5,2), r["residual"],  "Spectral Residual", "hot")
            _show(_ax(5,3), r["peaks"],     "Periodic Anomalies")
            _flag(_ax(5,3), "dft")

        # Row 6: Histogram + intensity transforms
        r = self.results.get("histogram", {})
        if r:
            ax_h = _ax(6,0)
            ax_h.set_facecolor("#1c1c1c")
            ax_h.bar(range(256), r["hist"], color="#4da6ff", width=1)
            ax_h.set_title("Histogram", fontsize=7, color="white", pad=2)
            ax_h.tick_params(colors="white", labelsize=5)
            ax_h.set_xlim(0,255)
            for sp in ax_h.spines.values(): sp.set_color("#333")
            _show(_ax(6,1), r["log_img"],   "Log Transform")
            _show(_ax(6,2), r["gamma_img"], "Gamma (γ=0.5)")
            _show(_ax(6,3), r["clahe_img"], "CLAHE Enhanced")
            _flag(_ax(6,3), "histogram")

        # Row 7: Spatial filtering
        r = self.results.get("spatial", {})
        if r:
            _show(_ax(7,0), r["gauss_residual"],  "Gauss Residual")
            _show(_ax(7,1), r["median_residual"],  "Median Residual")
            _show(_ax(7,2), r["laplacian"],         "Laplacian")
            _show(_ax(7,3), r["sharpness_anom"],    "Sharpness Anomaly", "hot")
            _flag(_ax(7,3), "spatial")

        # Row 8: Color spaces
        r = self.results.get("color", {})
        if r:
            _show(_ax(8,0), self.rgb,   "Original RGB")
            _show(_ax(8,1), r["hue"],   "Hue Channel (HSV)",  "hsv")
            _show(_ax(8,2), r["cb"],    "Cb (YCbCr)",         "cool")
            _show(_ax(8,3), r["cr"],    "Cr (YCbCr)",         "cool")
            _flag(_ax(8,3), "color")

        # Row 9: Morphology
        r = self.results.get("morph", {})
        if r:
            _show(_ax(9,0), r["eroded"],    "Eroded")
            _show(_ax(9,1), r["opened"],    "Opened")
            _show(_ax(9,2), r["boundary"],  "Boundary Extracted")
            _show(_ax(9,3), r["top_hat"],   "Top-Hat (Grayscale)", "hot")
            _flag(_ax(9,3), "morph")

        # Row 10: Morphology cont.
        if r:
            _show(_ax(10,0), r["filled"],    "Hole-Filled")
            _show(_ax(10,1), r["black_hat"], "Black-Hat", "hot")
            _show(_ax(10,2), r["morph_grad"],"Morph Gradient")
            # Connected components colourmap
            lmap = (r["label_map"] % 20 + 1) * (r["label_map"] > 0)
            _show(_ax(10,3), lmap, f"Connected Components (n={r['n_labels']-1})", "tab20")
            _flag(_ax(10,3), "morph")

        # Row 11: Watershed
        r = self.results.get("watershed", {})
        if r:
            _show(_ax(11,0), r["dist"],     "Distance Transform")
            _show(_ax(11,1), r["gradient"], "Gradient (for WS)")
            _show(_ax(11,2,2), r["seg_vis"], "Watershed Segmentation")
            _flag(_ax(11,2,2), "watershed")

        # Row 12: GLCM
        r = self.results.get("glcm", {})
        if r:
            for i, name in enumerate(r["names"]):
                ax = _ax(12, i)
                ax.set_facecolor("#1c1c1c")
                ax.plot(r["feats"][:, i], color="#4da6ff", lw=0.7)
                ax.axhline(r["feats"][:,i].mean(), color="#ff6b6b", lw=0.8, ls="--")
                ax.set_title(f"GLCM: {name}", fontsize=7, color="white", pad=2)
                ax.tick_params(colors="white", labelsize=5)
                for sp in ax.spines.values(): sp.set_color("#333")
            _flag(_ax(12,3), "glcm")

        # Row 12 cont: boundary descriptors + contours
        if r and r["descriptors"]:
            ax_b = _ax(12,3) if not r else _ax(12,3)  # reuse last axis
        r_glcm = self.results.get("glcm",{})
        if r_glcm:
            _show(_ax(13,0,2), r_glcm["contour_vis"], "Boundary Contours")

        # Row 13: verdict bar
        ax_v = fig.add_subplot(gs[13, 2:])
        ax_v.set_facecolor("#1c1c1c"); ax_v.set_xlim(0,1); ax_v.set_ylim(0,1); ax_v.axis("off")
        frac = min(ws, 1.0)
        ax_v.add_patch(plt.Rectangle((0.03,0.2), 0.94, 0.6, color="#2a2a2a"))
        ax_v.add_patch(plt.Rectangle((0.03,0.2), 0.94*frac, 0.6, color=vcolour))
        ax_v.text(0.5, 0.5, f"{verdict}  ({ws*100:.1f}/100)",
                  ha="center", va="center", fontsize=11, color="white",
                  fontweight="bold", fontfamily="monospace")

        if save_path:
            os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else ".", exist_ok=True)
            plt.savefig(save_path, dpi=110, bbox_inches="tight", facecolor="#0d0d0d")
            print(f"Dashboard saved → {save_path}")
        if show:
            plt.show()
        plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
# CONVENIENCE RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def analyse(image_path: str, save_path: str = None, show: bool = True):
    """Run full forensic analysis on a single image."""
    print(f"\nImage: {image_path}\n")
    det = ImageForensicsDetector(image_path)
    verdict, confidence, scores = det.run_all(verbose=True)
    det.plot_dashboard(save_path=save_path, show=show)
    return verdict, confidence, scores


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys
    img = sys.argv[1] if len(sys.argv) > 1 else "Data/Cat_Test.jpg"
    out = sys.argv[2] if len(sys.argv) > 2 else "Outputs/forensic_dashboard.png"
    analyse(img, save_path=out, show=False)
    print(f"\nDone. Dashboard saved to: {out}")


## Step 3 — Write Batch Evaluator

In [ ]:
%%writefile batch_evaluate.py
"""
Batch Evaluation — Columbia Uncompressed Image Splicing Detection Dataset
=========================================================================
Runs the forensic detector on ALL images in the dataset and reports:
  - Rule-based detector accuracy (weighted score threshold)
  - Logistic Regression fusion accuracy (trained on 80%, tested on 20%)
  - Accuracy, Precision, Recall, F1, Confusion Matrix for both methods
  - Per-image verdict table

Usage:
  python batch_evaluate.py                          # uses Data/ folder
  python batch_evaluate.py --data path/to/dataset   # custom path
  python batch_evaluate.py --limit 20               # quick test on 20 images
"""

import os
import sys
import argparse
import time
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from image_forensics import ImageForensicsDetector

warnings.filterwarnings("ignore")

TECHNIQUE_KEYS = ["ela", "jpeg", "noise", "edge", "copy_move",
                  "dft", "histogram", "spatial", "color", "morph", "watershed", "glcm"]

TECHNIQUE_LABELS = ["ELA", "JPEG", "Noise", "Edge+Hough", "Copy-Move",
                    "DFT", "Histogram", "Spatial", "Color", "Morphology", "Watershed", "GLCM"]

WEIGHTS = {"ela": 0.20, "jpeg": 0.15, "noise": 0.15, "edge": 0.10, "copy_move": 0.10,
           "dft": 0.05, "histogram": 0.05, "spatial": 0.05, "color": 0.05,
           "morph": 0.03, "watershed": 0.04, "glcm": 0.03}


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _collect_images(data_dir):
    exts = {".tif", ".jpg", ".jpeg", ".png", ".bmp"}

    def images_in(folder):
        if not os.path.isdir(folder):
            return []
        return sorted([os.path.join(folder, f) for f in os.listdir(folder)
                       if os.path.splitext(f)[1].lower() in exts])

    auth_dir = os.path.join(data_dir, "4cam_auth")
    splc_dir = os.path.join(data_dir, "4cam_splc")

    if not os.path.isdir(auth_dir) or not os.path.isdir(splc_dir):
        for sub in os.listdir(data_dir):
            sl = sub.lower()
            if "auth" in sl:
                auth_dir = os.path.join(data_dir, sub)
            if any(k in sl for k in ("splc", "tamp", "forg")):
                splc_dir = os.path.join(data_dir, sub)

    if not (os.path.isdir(auth_dir) and os.path.isdir(splc_dir)):
        raise FileNotFoundError(
            f"Cannot find auth/splc folders in '{data_dir}'.\n"
            "Expected: 4cam_auth/ and 4cam_splc/ (Columbia dataset structure)."
        )
    return images_in(auth_dir), images_in(splc_dir)


def _rule_verdict(scores_dict):
    ws = sum(scores_dict.get(k, 0) * WEIGHTS[k] for k in WEIGHTS)
    n  = sum(1 for k in WEIGHTS if scores_dict.get(k, 0) > 0.5)
    if ws > 0.55 or n >= 6:
        return "LIKELY TAMPERED", ws * 100
    elif ws > 0.30 or n >= 4:
        return "SUSPICIOUS", ws * 100
    else:
        return "LIKELY AUTHENTIC", ws * 100


def _verdict_to_label(verdict):
    return 1 if verdict in ("LIKELY TAMPERED", "SUSPICIOUS") else 0


# ─────────────────────────────────────────────────────────────────────────────
# BATCH RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_batch(data_dir="Data", limit=None, save_dir="Outputs", verbose=True):
    auth_paths, tamp_paths = _collect_images(data_dir)

    if limit:
        half = limit // 2
        auth_paths = auth_paths[:half]
        tamp_paths = tamp_paths[:limit - half]

    total = len(auth_paths) + len(tamp_paths)
    print(f"\n{'═'*62}")
    print(f"  BATCH EVALUATION — Columbia Uncompressed Splicing Dataset")
    print(f"  Authentic : {len(auth_paths)} images")
    print(f"  Tampered  : {len(tamp_paths)} images")
    print(f"  Total     : {total} images")
    print(f"{'═'*62}\n")

    os.makedirs(save_dir, exist_ok=True)
    results = []

    def process(paths, true_label, label_name):
        for i, path in enumerate(paths, 1):
            fname = os.path.basename(path)
            print(f"  [{label_name}] ({i}/{len(paths)}) {fname} ...", end=" ", flush=True)
            t0 = time.time()
            try:
                det = ImageForensicsDetector(path)
                det.run_all(verbose=False)
                scores_dict = det.scores
                verdict, confidence = _rule_verdict(scores_dict)
                pred_label = _verdict_to_label(verdict)
                elapsed = time.time() - t0
                correct = (pred_label == true_label)
                print(f"{'✓' if correct else '✗'}  [{verdict}]  {confidence:.1f}/100  ({elapsed:.1f}s)")
                results.append(dict(
                    path=path, fname=fname,
                    true_label=true_label, true_name=label_name,
                    verdict=verdict, pred_label=pred_label,
                    confidence=confidence, scores=scores_dict,
                    correct=correct, elapsed=elapsed
                ))
            except Exception as e:
                print(f"ERROR: {e}")
                results.append(dict(
                    path=path, fname=fname,
                    true_label=true_label, true_name=label_name,
                    verdict="ERROR", pred_label=-1,
                    confidence=0, scores={}, correct=False, elapsed=0
                ))

    process(auth_paths, true_label=0, label_name="AUTH ")
    process(tamp_paths, true_label=1, label_name="TAMP ")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────

def compute_metrics(y_true, y_pred):
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    n  = len(y_true)
    acc  = (tp + tn) / n * 100 if n > 0 else 0
    prec = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, accuracy=acc,
                precision=prec, recall=rec, f1=f1, n=n)


# ─────────────────────────────────────────────────────────────────────────────
# LOGISTIC REGRESSION FUSION
# ─────────────────────────────────────────────────────────────────────────────

def train_lr_fusion(results, test_size=0.2, random_state=42):
    """
    Uses the 12 detector scores as features and trains a Logistic Regression
    classifier to fuse them optimally.

    Returns:
        lr_model, scaler, train_idx, test_idx,
        train_metrics dict, test_metrics dict,
        lr_verdicts list (aligned to results)
    """
    valid = [(i, r) for i, r in enumerate(results) if r["pred_label"] >= 0]
    if len(valid) < 10:
        print("Not enough valid results to train LR (need ≥ 10).")
        return None, None, None, None, None, None, [r["pred_label"] for r in results]

    idxs   = [i for i, _ in valid]
    X      = np.array([[r["scores"].get(k, 0) for k in TECHNIQUE_KEYS] for _, r in valid])
    y      = np.array([r["true_label"] for _, r in valid])

    X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
        X, y, idxs, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler   = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_te_sc  = scaler.transform(X_te)

    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=random_state)
    lr.fit(X_tr_sc, y_tr)

    y_pred_tr = lr.predict(X_tr_sc)
    y_pred_te = lr.predict(X_te_sc)

    train_metrics = compute_metrics(y_tr, y_pred_tr)
    test_metrics  = compute_metrics(y_te, y_pred_te)

    # Build full predictions array (rule-based where LR wasn't tested)
    lr_preds = np.array([r["pred_label"] for r in results])
    for pos, orig_i in enumerate(idx_te):
        lr_preds[orig_i] = int(y_pred_te[pos])

    print(f"\n{'─'*62}")
    print(f"  LOGISTIC REGRESSION FUSION  (train={len(y_tr)}, test={len(y_te)})")
    print(f"  Train accuracy : {train_metrics['accuracy']:.1f}%")
    print(f"  Test  accuracy : {test_metrics['accuracy']:.1f}%")
    print(f"  Test  F1 score : {test_metrics['f1']:.1f}%")

    # Learned feature weights
    print(f"\n  Learned feature importances (logistic regression coefficients):")
    coefs = lr.coef_[0]
    for k, label, coef in sorted(zip(TECHNIQUE_KEYS, TECHNIQUE_LABELS, coefs),
                                   key=lambda x: abs(x[2]), reverse=True):
        bar = '█' * int(abs(coef) * 20)
        sign = '+' if coef > 0 else '-'
        print(f"    {label:<15s}  {sign}{abs(coef):.3f}  {bar}")
    print(f"{'─'*62}")

    return lr, scaler, idx_tr, idx_te, train_metrics, test_metrics, lr_preds.tolist()


# ─────────────────────────────────────────────────────────────────────────────
# PRINT SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

def print_summary(results, lr_preds=None):
    valid   = [r for r in results if r["pred_label"] >= 0]
    y_true  = np.array([r["true_label"] for r in valid])
    y_rule  = np.array([r["pred_label"] for r in valid])

    rule_m  = compute_metrics(y_true, y_rule)

    print(f"\n{'═'*62}")
    print(f"  RESULTS — RULE-BASED DETECTOR")
    print(f"{'═'*62}")
    print(f"  Images processed  : {rule_m['n']}")
    print(f"  Accuracy          : {rule_m['accuracy']:.1f}%")
    print(f"  Precision         : {rule_m['precision']:.1f}%")
    print(f"  Recall            : {rule_m['recall']:.1f}%")
    print(f"  F1 Score          : {rule_m['f1']:.1f}%")
    print(f"  TP={rule_m['tp']}  TN={rule_m['tn']}  FP={rule_m['fp']}  FN={rule_m['fn']}")

    if lr_preds is not None:
        y_lr = np.array([lr_preds[i] for i, r in enumerate(results) if r["pred_label"] >= 0])
        lr_m = compute_metrics(y_true, y_lr)
        print(f"\n{'─'*62}")
        print(f"  RESULTS — LOGISTIC REGRESSION FUSION (test set)")
        print(f"{'─'*62}")
        print(f"  Accuracy   : {lr_m['accuracy']:.1f}%")
        print(f"  Precision  : {lr_m['precision']:.1f}%")
        print(f"  Recall     : {lr_m['recall']:.1f}%")
        print(f"  F1 Score   : {lr_m['f1']:.1f}%")
        print(f"  TP={lr_m['tp']}  TN={lr_m['tn']}  FP={lr_m['fp']}  FN={lr_m['fn']}")
    print(f"{'═'*62}\n")

    sorted_r = sorted(enumerate(results), key=lambda x: x[1]["confidence"], reverse=True)
    print(f"  {'File':<38} {'True':>8} {'Verdict':<18} {'Score':>5}")
    print(f"  {'-'*38} {'-'*8} {'-'*18} {'-'*5}")
    for orig_i, r in sorted_r:
        if r["pred_label"] < 0:
            continue
        true_str = "TAMPERED" if r["true_label"] == 1 else "AUTHENTIC"
        ok = "✓" if r["correct"] else "✗"
        print(f"  {r['fname']:<38} {true_str:>8} {r['verdict']:<18} {r['confidence']:>4.0f}  {ok}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

def plot_results(results, lr_preds=None, lr_test_metrics=None,
                 save_path="Outputs/batch_results.png"):

    valid      = [r for r in results if r["pred_label"] >= 0]
    y_true     = np.array([r["true_label"] for r in valid])
    y_rule     = np.array([r["pred_label"] for r in valid])
    rule_m     = compute_metrics(y_true, y_rule)

    auth_scores = [r["confidence"] for r in valid if r["true_label"] == 0]
    tamp_scores = [r["confidence"] for r in valid if r["true_label"] == 1]

    has_lr = (lr_preds is not None and lr_test_metrics is not None)

    fig = plt.figure(figsize=(20, 13))
    fig.patch.set_facecolor("#0d0d0d")
    rows = 3 if has_lr else 2
    gs   = gridspec.GridSpec(rows, 3, figure=fig,
                             hspace=0.50, wspace=0.35,
                             top=0.91, bottom=0.06, left=0.06, right=0.97)

    fig.text(0.5, 0.96,
             "BATCH EVALUATION — Columbia Uncompressed Image Splicing Detection Dataset",
             ha="center", fontsize=15, color="white", fontweight="bold", fontfamily="monospace")
    fig.text(0.5, 0.925,
             f"Rule-based:  Acc={rule_m['accuracy']:.1f}%  "
             f"Prec={rule_m['precision']:.1f}%  "
             f"Recall={rule_m['recall']:.1f}%  "
             f"F1={rule_m['f1']:.1f}%   |   n={rule_m['n']} images",
             ha="center", fontsize=11, color="#aaaaaa")

    # ── 1. Score distributions ─────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor("#1c1c1c")
    bins = np.linspace(0, 100, 21)
    ax1.hist(auth_scores, bins=bins, alpha=0.75, color="#44ff88", label="Authentic")
    ax1.hist(tamp_scores, bins=bins, alpha=0.75, color="#ff4444", label="Tampered")
    ax1.axvline(30, color="#ffaa00", ls="--", lw=1.2, label="SUSPICIOUS (30)")
    ax1.axvline(55, color="#ff6666", ls="--", lw=1.2, label="TAMPERED (55)")
    ax1.set_title("Suspicion Score Distributions", color="white", fontsize=11)
    ax1.set_xlabel("Suspicion Score (0–100)", color="#aaa")
    ax1.set_ylabel("Image count", color="#aaa")
    ax1.tick_params(colors="white")
    ax1.legend(fontsize=8, facecolor="#1c1c1c", labelcolor="white", edgecolor="#555")
    for sp in ax1.spines.values(): sp.set_color("#333")

    # ── 2. Confusion matrix — rule-based ──────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor("#1c1c1c")
    cm = np.array([[rule_m["tn"], rule_m["fp"]], [rule_m["fn"], rule_m["tp"]]])
    ax2.imshow(cm, cmap="Blues", vmin=0)
    ax2.set_xticks([0, 1]); ax2.set_yticks([0, 1])
    ax2.set_xticklabels(["Pred: AUTH", "Pred: TAMP"], color="white", fontsize=9)
    ax2.set_yticklabels(["True: AUTH", "True: TAMP"], color="white", fontsize=9)
    for i in range(2):
        for j in range(2):
            ax2.text(j, i, str(cm[i, j]), ha="center", va="center",
                     fontsize=18, fontweight="bold",
                     color="white" if cm[i, j] < cm.max() * 0.6 else "black")
    ax2.set_title("Confusion Matrix (Rule-based)", color="white", fontsize=11)
    ax2.tick_params(colors="white")

    # ── 3. Metrics comparison ──────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.set_facecolor("#1c1c1c")
    metric_names = ["Accuracy", "Precision", "Recall", "F1"]
    rule_vals    = [rule_m["accuracy"], rule_m["precision"], rule_m["recall"], rule_m["f1"]]
    x = np.arange(len(metric_names))
    if has_lr:
        lr_vals = [lr_test_metrics["accuracy"], lr_test_metrics["precision"],
                   lr_test_metrics["recall"],   lr_test_metrics["f1"]]
        bars1 = ax3.bar(x - 0.2, rule_vals, 0.35, color="#4da6ff", alpha=0.85, label="Rule-based")
        bars2 = ax3.bar(x + 0.2, lr_vals,   0.35, color="#ff9944", alpha=0.85, label="Logistic Reg.")
        for bar, val in list(zip(bars1, rule_vals)) + list(zip(bars2, lr_vals)):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                     f"{val:.0f}%", ha="center", va="bottom", color="white", fontsize=8)
        ax3.legend(fontsize=9, facecolor="#1c1c1c", labelcolor="white", edgecolor="#555")
    else:
        colours = ["#4da6ff", "#44ff88", "#ffaa00", "#ff6b6b"]
        bars = ax3.bar(x, rule_vals, 0.5, color=colours)
        for bar, val in zip(bars, rule_vals):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                     f"{val:.0f}%", ha="center", va="bottom", color="white", fontsize=9)
    ax3.set_xticks(x); ax3.set_xticklabels(metric_names, color="white", fontsize=9)
    ax3.set_ylim(0, 115); ax3.set_title("Performance Metrics (%)", color="white", fontsize=11)
    ax3.set_ylabel("Score (%)", color="#aaa"); ax3.tick_params(colors="white")
    ax3.axhline(100, color="#333", ls="--", lw=0.5)
    for sp in ax3.spines.values(): sp.set_color("#333")

    # ── 4. Per-image scatter ───────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, :2])
    ax4.set_facecolor("#1c1c1c")
    sorted_r = sorted(valid, key=lambda x: x["confidence"], reverse=True)
    for xi, r in enumerate(sorted_r):
        colour = "#ff4444" if r["true_label"] == 1 else "#44ff88"
        marker = "o" if r["correct"] else "x"
        ax4.scatter(xi, r["confidence"], color=colour, marker=marker, s=20, alpha=0.8)
    ax4.axhline(55, color="#ff6666", ls="--", lw=1, label="TAMPERED threshold (55)")
    ax4.axhline(30, color="#ffaa00", ls="--", lw=1, label="SUSPICIOUS threshold (30)")
    ax4.set_title("Per-Image Suspicion Scores  (red=tampered, green=authentic, ✗=wrong prediction)",
                  color="white", fontsize=10)
    ax4.set_xlabel("Images ranked by suspicion score", color="#aaa")
    ax4.set_ylabel("Suspicion score", color="#aaa")
    ax4.tick_params(colors="white"); ax4.set_xlim(-1, len(sorted_r))
    ax4.legend(fontsize=8, facecolor="#1c1c1c", labelcolor="white", edgecolor="#555")
    for sp in ax4.spines.values(): sp.set_color("#333")

    # ── 5. Technique trigger rates ─────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.set_facecolor("#1c1c1c")
    auth_rates = [np.mean([r["scores"].get(k, 0) for r in valid if r["true_label"] == 0])
                  for k in TECHNIQUE_KEYS]
    tamp_rates = [np.mean([r["scores"].get(k, 0) for r in valid if r["true_label"] == 1])
                  for k in TECHNIQUE_KEYS]
    y_pos = np.arange(len(TECHNIQUE_KEYS))
    ax5.barh(y_pos - 0.2, auth_rates, 0.35, color="#44ff88", alpha=0.85, label="Authentic")
    ax5.barh(y_pos + 0.2, tamp_rates, 0.35, color="#ff4444", alpha=0.85, label="Tampered")
    ax5.set_yticks(y_pos)
    ax5.set_yticklabels(TECHNIQUE_LABELS, color="white", fontsize=8)
    ax5.set_title("Avg Technique Score by Class", color="white", fontsize=10)
    ax5.set_xlabel("Mean suspicion score (0–1)", color="#aaa")
    ax5.tick_params(colors="white")
    ax5.legend(fontsize=8, facecolor="#1c1c1c", labelcolor="white", edgecolor="#555")
    for sp in ax5.spines.values(): sp.set_color("#333")

    # ── 6. LR confusion matrix + feature weights ───────────────────────────
    if has_lr:
        y_lr    = np.array([lr_preds[i] for i, r in enumerate(results) if r["pred_label"] >= 0])
        lr_full = compute_metrics(y_true, y_lr)
        cm_lr   = np.array([[lr_full["tn"], lr_full["fp"]], [lr_full["fn"], lr_full["tp"]]])

        ax6 = fig.add_subplot(gs[2, 0])
        ax6.set_facecolor("#1c1c1c")
        ax6.imshow(cm_lr, cmap="Oranges", vmin=0)
        ax6.set_xticks([0, 1]); ax6.set_yticks([0, 1])
        ax6.set_xticklabels(["Pred: AUTH", "Pred: TAMP"], color="white", fontsize=9)
        ax6.set_yticklabels(["True: AUTH", "True: TAMP"], color="white", fontsize=9)
        for i in range(2):
            for j in range(2):
                ax6.text(j, i, str(cm_lr[i, j]), ha="center", va="center",
                         fontsize=18, fontweight="bold",
                         color="white" if cm_lr[i, j] < cm_lr.max() * 0.6 else "black")
        ax6.set_title(f"Confusion Matrix (Logistic Regression)\nAcc={lr_test_metrics['accuracy']:.1f}%  F1={lr_test_metrics['f1']:.1f}%",
                      color="white", fontsize=10)
        ax6.tick_params(colors="white")

        ax7 = fig.add_subplot(gs[2, 1:])
        ax7.set_facecolor("#1c1c1c")
        ax7.set_title("Logistic Regression — Learned Feature Weights\n(positive = evidence of tampering)",
                      color="white", fontsize=10)
        # We don't have lr object here, so use the score deltas as proxy
        diffs = [(t - a, lbl) for t, a, lbl in zip(tamp_rates, auth_rates, TECHNIQUE_LABELS)]
        diffs.sort(key=lambda x: x[0])
        vals  = [d[0] for d in diffs]
        lbls  = [d[1] for d in diffs]
        colors_bar = ["#ff4444" if v > 0 else "#44ff88" for v in vals]
        ax7.barh(range(len(vals)), vals, color=colors_bar, alpha=0.85)
        ax7.set_yticks(range(len(lbls))); ax7.set_yticklabels(lbls, color="white", fontsize=9)
        ax7.axvline(0, color="#888", lw=0.8)
        ax7.set_xlabel("Mean score difference (tampered − authentic)", color="#aaa")
        ax7.tick_params(colors="white")
        for sp in ax7.spines.values(): sp.set_color("#333")

    plt.savefig(save_path, dpi=110, bbox_inches="tight", facecolor="#0d0d0d")
    print(f"\nResults plot saved → {save_path}")
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data",  default="Data",    help="Path to dataset folder")
    parser.add_argument("--out",   default="Outputs", help="Output folder")
    parser.add_argument("--limit", type=int, default=None,
                        help="Limit total images (e.g. 20 for quick test)")
    args = parser.parse_args()

    results = run_batch(data_dir=args.data, limit=args.limit, save_dir=args.out)

    valid = [r for r in results if r["pred_label"] >= 0]
    if len(valid) == 0:
        print("No valid results."); return

    lr_model, scaler, idx_tr, idx_te, train_m, test_m, lr_preds = train_lr_fusion(results)

    print_summary(results, lr_preds=lr_preds)
    plot_results(results, lr_preds=lr_preds, lr_test_metrics=test_m,
                 save_path=os.path.join(args.out, "batch_results.png"))


if __name__ == "__main__":
    main()


## Step 4 — Extract Columbia Dataset
Extracts the zip you uploaded. Skipped automatically if already extracted.

In [ ]:
import os, zipfile, glob

def extract_dataset():
    # Already extracted?
    if os.path.isdir('Data/4cam_auth') and os.path.isdir('Data/4cam_splc'):
        auth_n = len([f for f in os.listdir('Data/4cam_auth') if f.endswith('.tif')])
        splc_n = len([f for f in os.listdir('Data/4cam_splc') if f.endswith('.tif')])
        print(f"Dataset ready: {auth_n} authentic  {splc_n} spliced")
        return True

    # Find the zip
    candidates = (glob.glob('*.zip') + glob.glob('columbia*.zip') +
                  glob.glob('dvmm*.zip') + glob.glob('/content/*.zip'))
    candidates = list(dict.fromkeys(candidates))  # deduplicate

    zip_path = None
    for c in candidates:
        if os.path.exists(c) and os.path.getsize(c) > 100_000:
            zip_path = c
            break

    if zip_path is None:
        print("ZIP not found. Did you upload dvmm_splc_uncompressed.zip?")
        print("Files in /content:", os.listdir('/content') if os.path.exists('/content') else os.listdir('.'))
        return False

    print(f"Extracting {zip_path} ...")
    os.makedirs('Data', exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('Data/')

    # Some zips nest an extra folder — find where tifs landed
    for root, dirs, files in os.walk('Data'):
        tifs = [f for f in files if f.endswith('.tif')]
        if tifs:
            print(f"Found TIFs in: {root}")
            break

    if os.path.isdir('Data/4cam_auth') and os.path.isdir('Data/4cam_splc'):
        auth_n = len([f for f in os.listdir('Data/4cam_auth') if f.endswith('.tif')])
        splc_n = len([f for f in os.listdir('Data/4cam_splc') if f.endswith('.tif')])
        print(f"Dataset ready: {auth_n} authentic  {splc_n} spliced  (total={auth_n+splc_n})")
        return True
    else:
        print("Unexpected zip structure. Listing Data/:")
        for root, dirs, files in os.walk('Data'):
            print(f"  {root}/ — {len(files)} files")
        return False

dataset_ok = extract_dataset()

## Step 5 — Verify Test Image

In [ ]:
import os

IMAGE_PATH = 'Cat_Test2.jpg'   # Change if you uploaded a different image

if not os.path.exists(IMAGE_PATH):
    print(f"WARNING: {IMAGE_PATH} not found.")
    print("Files here:", [f for f in os.listdir('.') if f.lower().endswith(('.jpg','.jpeg','.png','.tif'))])
else:
    print(f"Image ready: {IMAGE_PATH}")

## Step 6 — Run All 12 Detectors on Test Image

In [ ]:
import importlib, sys
if 'image_forensics' in sys.modules:
    del sys.modules['image_forensics']
import image_forensics as ifm

os.makedirs('Outputs', exist_ok=True)
detector = ifm.ImageForensicsDetector(IMAGE_PATH)
verdict, confidence, scores = detector.run_all(verbose=True)

## Step 7 — Full Dashboard

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 90

detector.plot_dashboard(save_path='Outputs/forensic_dashboard.png', show=True)

---
## Deep-Dive — What Each Technique Finds in the Image
Each cell below shows the intermediate outputs of one technique.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def norm(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def show_row(imgs_titles_cmaps, suptitle, score_line, flagged):
    n = len(imgs_titles_cmaps)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), facecolor='#0f0f0f')
    if n == 1: axes = [axes]
    fig.suptitle(suptitle, color='white', fontsize=11)
    for ax, (img, title, cmap) in zip(axes, imgs_titles_cmaps):
        ax.set_facecolor('#1a1a1a')
        ax.imshow(img if (img.ndim==3 and cmap is None) else norm(img), cmap=cmap, aspect='auto')
        ax.set_title(title, color='white', fontsize=9); ax.axis('off')
    colour = '#ff6666' if flagged else '#66ff99'
    fig.text(0.5, 0.01, score_line, ha='center', color=colour, fontsize=10)
    plt.tight_layout(); plt.show()

print("Helper functions ready.")

### Technique 1 — ELA (Error Level Analysis)
Save image as JPEG at quality 95 → reload → compute pixel difference. Spliced regions show different re-compression error than the rest.

In [ ]:
r = detector.results['ela']
show_row([
    (detector.rgb, 'Original', None),
    (r['gray'],    'ELA Map (brighter = higher error)', 'hot'),
    (r['vis'],     'ELA + CLAHE (anomalies amplified)', 'hot'),
], '1. ELA — Error Level Analysis  (DIP: Image Fundamentals + Coding)',
   f"{'FLAGGED' if detector.detected['ela'] else 'clear'}  suspicion={r['suspicion_pct']:.2f}%  score={detector.scores['ela']:.3f}",
   detector.detected['ela'])

### Technique 2 — JPEG Artifact Detection
JPEG compresses in 8×8 blocks. Tampered regions break normal block statistics. Otsu's method automatically thresholds suspicious blocks.

In [ ]:
r = detector.results['jpeg']
show_row([
    (r['std_map'],  'Block Std-Dev (8×8)', 'hot'),
    (r['edge_map'], 'Block Edge Density',  'hot'),
    (r['artifact'], 'Combined Artifact',   'gray'),
    (r['suspicious'].astype(np.uint8)*255, f'Otsu Suspicious Blocks', 'hot'),
], '2. JPEG Artifact Detection  (DIP: Image Coding + Segmentation)',
   f"{'FLAGGED' if detector.detected['jpeg'] else 'clear'}  score={detector.scores['jpeg']:.3f}  otsu_thresh={r['otsu_val']:.3f}",
   detector.detected['jpeg'])

### Technique 3 — Wiener Filter + K-means Noise
Wiener denoising at 3 scales → noise residue reveals camera noise pattern → K-means clusters separate normal vs anomalous noise regions.

In [ ]:
r = detector.results['noise']
show_row([
    (r['noise_map'],                         'Noise Residue (3-scale Wiener)', 'gray'),
    (r['clusters'].astype(np.float32),       'K-means Clusters (k=3)',         'tab10'),
    (r['suspect_mask'].astype(np.uint8)*255, 'Anomalous Region',               'hot'),
], '3. Wiener Filter + K-means  (DIP: Image Restoration)',
   f"{'FLAGGED' if detector.detected['noise'] else 'clear'}  score={detector.scores['noise']:.3f}",
   detector.detected['noise'])

### Technique 4 — Edge Detection + Hough Transform
Canny / Sobel / Prewitt edges. Edges >2σ from the image's own median are flagged. Hough detects geometric line artifacts from splicing.

In [ ]:
r = detector.results['edge']
show_row([
    (r['canny'],   'Canny Edges', 'gray'),
    (r['mag'],     'Sobel Magnitude', 'gray'),
    (r['suspicious'].astype(np.uint8)*255, 'Anomalous Edges (>2σ)', 'hot'),
    (r['hough_img'], f'Hough Lines ({r["n_lines"]} detected)', None),
], '4. Edge Detection + Hough Transform  (DIP: Enhancement + Segmentation)',
   f"{'FLAGGED' if detector.detected['edge'] else 'clear'}  score={detector.scores['edge']:.3f}  lines={r['n_lines']}",
   detector.detected['edge'])

### Technique 5 — Copy-Move Detection
DCT features from overlapping 16×16 patches are compared. Identical patches far apart = cloned region.

In [ ]:
r = detector.results['copy_move']
show_row([
    (detector.rgb, 'Original', None),
    (r['map'].astype(np.uint8)*255, f'Copy-Move Map ({r["matched"]} matched pairs)', 'hot'),
], '5. Copy-Move Detection (DCT patch matching)  (DIP: Image Coding + Representation)',
   f"{'FLAGGED' if detector.detected['copy_move'] else 'clear'}  score={detector.scores['copy_move']:.3f}  matched={r['matched']}",
   detector.detected['copy_move'])

### Technique 6 — DFT Frequency Spectrum
Authentic images have a smooth 1/f spectrum. Spliced content, resampling, or resizing introduces periodic spectral peaks.

In [ ]:
r = detector.results['dft']
show_row([
    (r['magnitude'], 'Magnitude Spectrum', 'inferno'),
    (r['phase'],     'Phase Spectrum',     'twilight'),
    (r['residual'],  'Spectral Residual',  'hot'),
    (r['peaks'].astype(np.uint8)*255, 'Periodic Anomaly Peaks', 'hot'),
], '6. DFT Frequency Analysis  (DIP: Image Transformation)',
   f"{'FLAGGED' if detector.detected['dft'] else 'clear'}  score={detector.scores['dft']:.3f}",
   detector.detected['dft'])

### Technique 7 — Histogram + Intensity Transforms
Comb gaps in the histogram indicate stretching (post-edit). Also shows log transform, gamma correction, and CLAHE.

In [ ]:
r = detector.results['histogram']
fig, axes = plt.subplots(1, 4, figsize=(18, 4), facecolor='#0f0f0f')
fig.suptitle('7. Histogram + Log/Gamma/CLAHE  (DIP: Fundamentals + Enhancement)', color='white', fontsize=11)
axes[0].set_facecolor('#1a1a1a')
axes[0].bar(range(256), r['hist'], color='#4da6ff', width=1)
axes[0].set_title(f'Pixel Histogram (gap_ratio={r["gap_ratio"]:.2f})', color='white', fontsize=9)
axes[0].tick_params(colors='white'); axes[0].set_xlim(0,255)
for sp in axes[0].spines.values(): sp.set_color('#333')
for ax, (img, title, cmap) in zip(axes[1:], [
    (r['log_img'],   'Log Transform', 'gray'),
    (r['gamma_img'], 'Gamma (γ=0.5)', 'gray'),
    (r['clahe_img'], 'CLAHE Enhanced', 'gray'),
]):
    ax.set_facecolor('#1a1a1a'); ax.imshow(norm(img), cmap=cmap, aspect='auto')
    ax.set_title(title, color='white', fontsize=9); ax.axis('off')
colour = '#ff6666' if detector.detected['histogram'] else '#66ff99'
fig.text(0.5,0.01,f"{'FLAGGED' if detector.detected['histogram'] else 'clear'}  gap={r['gap_ratio']:.3f}  clip={r['clip_score']:.3f}",
         ha='center', color=colour, fontsize=10)
plt.tight_layout(); plt.show()

### Technique 8 — Spatial Filtering
Gaussian and Median filter residuals reveal sharpness inconsistencies. Laplacian highlights fine detail differences. A pasted object from a different source often has a different blur level.

In [ ]:
r = detector.results['spatial']
show_row([
    (r['gauss_residual'],  'Gaussian Residual', 'gray'),
    (r['median_residual'], 'Median Residual',   'gray'),
    (r['laplacian'],       'Laplacian Response','gray'),
    (r['sharpness_anom'].astype(np.uint8)*255, 'Sharpness Anomaly', 'hot'),
], '8. Spatial Filtering  (DIP: Image Enhancement)',
   f"{'FLAGGED' if detector.detected['spatial'] else 'clear'}  score={detector.scores['spatial']:.3f}",
   detector.detected['spatial'])

### Technique 9 — Color Space Analysis
Local window statistics across RGB, HSV, and YCbCr. Inconsistent colour distribution between windows = possible region from different source/lighting.

In [ ]:
r = detector.results['color']
show_row([
    (detector.rgb, 'Original (RGB)', None),
    (r['hue'],     'Hue (HSV)',      'hsv'),
    (r['cb'],      'Cb (YCbCr)',     'cool'),
    (r['cr'],      'Cr (YCbCr)',     'cool'),
], '9. Color Space Analysis  (DIP: Color Image Processing)',
   f"{'FLAGGED' if detector.detected['color'] else 'clear'}  rgb={r['rgb_score']:.3f}  hsv={r['hsv_score']:.3f}  ycbcr={r['ycc_score']:.3f}",
   detector.detected['color'])

### Technique 10 — Morphological Analysis
Full pipeline: Otsu binarisation → erosion → dilation → opening → closing → boundary extraction → hole filling → connected components → top-hat → black-hat → morphological gradient.

In [ ]:
r = detector.results['morph']
fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor='#0f0f0f')
fig.suptitle('10. Morphological Analysis  (DIP: Morphological Image Processing)', color='white', fontsize=11)
for ax, (img, title, cmap) in zip(axes.flat, [
    (r['binary'],   'Binary (Otsu)',      'gray'),
    (r['eroded'],   'Erosion',            'gray'),
    (r['dilated'],  'Dilation',           'gray'),
    (r['opened'],   'Opening',            'gray'),
    (r['closed'],   'Closing',            'gray'),
    (r['boundary'], 'Boundary Extraction','gray'),
    (r['filled'],   'Hole Filling',       'gray'),
    (r['top_hat'],  'Top-Hat',            'hot'),
]):
    ax.set_facecolor('#1a1a1a'); ax.imshow(norm(img), cmap=cmap, aspect='auto')
    ax.set_title(title, color='white', fontsize=9); ax.axis('off')
colour = '#ff6666' if detector.detected['morph'] else '#66ff99'
fig.text(0.5,0.01,f"{'FLAGGED' if detector.detected['morph'] else 'clear'}  components={r['n_labels']-1}  score={detector.scores['morph']:.3f}",
         ha='center', color=colour, fontsize=10)
plt.tight_layout(); plt.show()

### Technique 11 — Watershed Segmentation
Distance transform finds region centres → watershed floods upward from those seeds. Abnormal region shapes or area variance flags tampering.

In [ ]:
r = detector.results['watershed']
show_row([
    (r['dist'],     'Distance Transform (seeds)', 'hot'),
    (r['gradient'], 'Gradient (watershed input)', 'gray'),
    (r['seg_vis'],  'Watershed Regions',           None),
], '11. Watershed Segmentation  (DIP: Image Segmentation)',
   f"{'FLAGGED' if detector.detected['watershed'] else 'clear'}  score={detector.scores['watershed']:.3f}",
   detector.detected['watershed'])

### Technique 12 — GLCM Texture + Boundary Descriptors
Haralick features (contrast, correlation, energy, homogeneity) in a sliding window. Windows far from the global mean are anomalous. Also computes area, perimeter, compactness, eccentricity for each boundary region.

In [ ]:
r = detector.results['glcm']
fig, axes = plt.subplots(1, 5, figsize=(22, 4), facecolor='#0f0f0f')
fig.suptitle('12. GLCM Texture + Boundary Descriptors  (DIP: Representation & Recognition)', color='white', fontsize=11)
for ax, name in zip(axes[:4], r['names']):
    ax.set_facecolor('#1a1a1a')
    vals = r['feats'][:, r['names'].index(name)]
    ax.plot(vals, color='#4da6ff', lw=0.8)
    ax.axhline(vals.mean(), color='#ff6b6b', lw=1.2, ls='--', label='mean')
    ax.set_title(f'GLCM: {name}', color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=7)
    for sp in ax.spines.values(): sp.set_color('#333')
    ax.legend(fontsize=7, facecolor='#1a1a1a', labelcolor='white', edgecolor='#444')
axes[4].set_facecolor('#1a1a1a')
axes[4].imshow(r['contour_vis']); axes[4].set_title('Boundary Contours', color='white', fontsize=9); axes[4].axis('off')
colour = '#ff6666' if detector.detected['glcm'] else '#66ff99'
desc_str = ''
if r['descriptors']:
    d = r['descriptors'][0]
    desc_str = f"  area={d['area']:.0f}  perim={d['perimeter']:.0f}  compact={d['compactness']:.3f}"
fig.text(0.5,0.01,f"{'FLAGGED' if detector.detected['glcm'] else 'clear'}  outlier_ratio={r['outlier_ratio']:.3f}{desc_str}",
         ha='center', color=colour, fontsize=10)
plt.tight_layout(); plt.show()

---
## Final Verdict

In [ ]:
weights = {'ela':0.20,'jpeg':0.15,'noise':0.15,'edge':0.10,'copy_move':0.10,
           'dft':0.05,'histogram':0.05,'spatial':0.05,'color':0.05,
           'morph':0.03,'watershed':0.04,'glcm':0.03}
labels  = {'ela':'ELA','jpeg':'JPEG Artifacts','noise':'Wiener Noise',
           'edge':'Edge+Hough','copy_move':'Copy-Move','dft':'DFT Spectrum',
           'histogram':'Histogram','spatial':'Spatial Filter',
           'color':'Color Spaces','morph':'Morphology',
           'watershed':'Watershed','glcm':'GLCM Texture'}
ws = sum(detector.scores.get(k,0)*w for k,w in weights.items())
n  = sum(detector.detected.values())
verdict = ('LIKELY TAMPERED' if ws > 0.55 or n >= 6 else
           'SUSPICIOUS'      if ws > 0.30 or n >= 4 else
           'LIKELY AUTHENTIC')
print('='*58)
print(f'  IMAGE     : {IMAGE_PATH}')
print(f'  VERDICT   : {verdict}')
print(f'  Suspicion : {ws*100:.1f} / 100')
print(f'  Triggered : {n}/12 detectors')
print('='*58)
for k in weights:
    flag = 'FLAGGED' if detector.detected.get(k) else 'clear  '
    print(f'  {flag}  {labels[k]:<20s}  score={detector.scores.get(k,0):.3f}  weight={weights[k]}')

## Download Single-Image Dashboard

In [ ]:
from google.colab import files
import os
if os.path.exists('Outputs/forensic_dashboard.png'):
    files.download('Outputs/forensic_dashboard.png')

---
## Part 2 — Batch Evaluation on the Columbia Dataset

Runs the detector on all ~363 labelled images.  
Trains a **Logistic Regression** on 80% of the dataset to learn optimal fusion weights.  
Tests on the remaining 20% and reports Accuracy / Precision / Recall / F1 / Confusion Matrix.

In [ ]:
# Confirm dataset is ready before running batch
import os
if os.path.isdir('Data/4cam_auth') and os.path.isdir('Data/4cam_splc'):
    auth_n = len([f for f in os.listdir('Data/4cam_auth') if f.endswith('.tif')])
    splc_n = len([f for f in os.listdir('Data/4cam_splc') if f.endswith('.tif')])
    print(f"Dataset: {auth_n} authentic  {splc_n} spliced  (total={auth_n+splc_n})")
    dataset_ok = True
else:
    print("Dataset not found. Make sure you uploaded the zip and Step 4 ran successfully.")
    dataset_ok = False

In [ ]:
# Run batch evaluation — all images, with LR fusion
# This takes 10-20 minutes for the full dataset.
import os
os.makedirs('Outputs', exist_ok=True)

if dataset_ok:
    !python batch_evaluate.py --data Data/ --out Outputs/
else:
    print("Skipped — dataset not ready.")

In [ ]:
# Show results plot (score distributions, confusion matrices, metrics comparison)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

plot_path = 'Outputs/batch_results.png'
if os.path.exists(plot_path):
    img = mpimg.imread(plot_path)
    fig, ax = plt.subplots(figsize=(20, 13))
    ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print("Results plot not found — make sure the batch evaluation above completed.")

In [ ]:
# Download batch results
from google.colab import files
import os
if os.path.exists('Outputs/batch_results.png'):
    files.download('Outputs/batch_results.png')